# CANDIDATE FEATURE IDENTIFICATION
The purpose of this notebook is to aggregate in one location the potential features to include in the training set. 
Inputs: External Dataset Parquet Files. 
Desired Outputs: 
- Of redundant variables, select the most reliable and informative one. 
- Understand how to calculate any computed features.

Notes
- Use a separate .py module under src/external_dataset_name/ingest.py to pull in the data with API or other method and write to data/interim/external_dataset_name.parquet. 
- This notebook should read in just a small sample of each dataset for feature exploration. Reading in the full dataset may crash the Jupyter kernel. 

For each feature, attempt to identify:
- time of observation
- unit of observation / what it actually is
- geographic scope (i.e. is it at the census block group, tract, or zip code level?)
- missingness
- what preprocessing is required (e.g. geocoding to census block group, converting a raw count to density)

In [33]:
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [34]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

## Lottery and Sales Data (for reference when merging)

In [35]:
lottery_locations_file_path = Path(".").resolve().parent / "data/final/location_data.parquet"

#locations_pq = pq.ParquetFile(lottery_locations_file_path)
#locations_first_100_rows = next(locations_pq.iter_batches(batch_size=100))
#locations_df = pa.Table.from_batches([locations_first_100_rows]).to_pandas()

locations_df = pd.read_parquet(str(lottery_locations_file_path))

In [43]:
print(locations_df.shape)
#locations_df.head()

test_retailer = locations_df[locations_df['agtno'] == '113165']
test_retailer

(16790, 14)


,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,region,competitors_within_1_mi
13970,113165,SMOKES 4 LESS,59 NORTH PLANK RD,NEWBURGH,12550,chain,COURIER,-74.03152,41.52378,b'\x01\x01\x00\x00\x00\xe0Jvl\x04\x82R\xc0B&\x199\x0b\xc3D@',360710103003,36071010300,Mid-Hudson,12


In [37]:
lottery_sales_file_path = Path(".").resolve().parent / "data/final/sales_data.parquet"

#sales_pq = pq.ParquetFile(lottery_sales_file_path)
#sales_first_100_rows = next(sales_pq.iter_batches(batch_size=1000))
#sales_df = pa.Table.from_batches([sales_first_100_rows]).to_pandas()

sales_df = pd.read_parquet(str(lottery_sales_file_path))

In [42]:
print(sales_df.shape)
print(sales_df['avg_daily_sales'].describe())

test_df = sales_df[sales_df['avg_daily_sales']>500000]
test_df

(14351, 15)
count     14351.000000
mean       2356.030566
std        5721.900850
min           0.413333
25%         665.106849
50%        1467.847945
75%        2888.675342
max      593051.376712
Name: avg_daily_sales, dtype: float64


,agtno,total_sales,first_date,last_date,first_pos_date,last_pos_date,pos_sales_days,num_cal_days,num_active_days,days_before_first_sale,days_after_last_sale,avg_cal_daily_sales,avg_active_daily_sales,use_active_period,avg_daily_sales
4984,113165,216463752.5,2025-01-01,2025-12-31,2025-01-01,2025-12-31,365.0,365,365.0,0.0,0.0,593051.376712,593051.376712,False,593051.376712


## Overture Maps Data

In [13]:
interim_overture_file_path = Path(".").resolve().parent / "data/interim/external/overture_filtered.parquet"

interim_overture_pq = pq.ParquetFile(interim_overture_file_path)
interim_overture_first_100_rows = next(interim_overture_pq.iter_batches(batch_size=5000))
interim_overture_df = pa.Table.from_batches([interim_overture_first_100_rows]).to_pandas()

interim_overture_df['highest_hierarchy'] = interim_overture_df['taxonomy_hierarchy'].str[0]
print(interim_overture_df['highest_hierarchy'].unique())
print(interim_overture_df['basic_category'].nunique())
interim_overture_df.sample(20)

['services_and_business' 'sports_and_recreation' 'education' 'health_care'
 'arts_and_entertainment' 'shopping' 'lifestyle_services'
 'travel_and_transportation' 'community_and_government'
 'cultural_and_historic' 'geographic_entities' 'food_and_drink' 'lodging'
 None]
201


,id,geometry,longitude,latitude,basic_category,taxonomy_primary,taxonomy_hierarchy,confidence,brand_name,first_seen,highest_hierarchy
1622,90ea3429-12ab-44d7-9afa-cd38c4c42104,b'\x01\x01\x00\x00\x00\xe4O\xaf\xef^)R\xc0\x9d\xdf(`XiD@',-72.646419,40.823010,health_care,health_care,[health_care],0.919912,NaN,2025-06-25.0,health_care
2017,935f4e80-6fa7-4227-b30a-27bd4937ccd2,"b""\x01\x01\x00\x00\x00\xbd\xe7\xd9\x0f\xa5,R\xc0\x8f'\xb4r\xd0hD@""",-72.697575,40.818861,personal_or_beauty_service,hair_salon,"[lifestyle_services, personal_or_beauty_service, hair_salon]",0.919912,NaN,2025-06-25.0,lifestyle_services
2668,ca4f6871-e19c-4800-9235-079df7af6246,b'\x01\x01\x00\x00\x00\xec\x82f|\x04;R\xc0\x9a\xb4\xa9\xbaGyD@',-72.922149,40.947502,food_and_beverage_store,beer_wine_spirits_store,"[shopping, food_and_beverage_store, beer_wine_spirits_store]",0.936253,NaN,2025-06-25.0,shopping
267,f61bda77-dfb2-4e8b-8467-d79620ad923f,b'\x01\x01\x00\x00\x00\xb1\xf3_\xb5l4R\xc0s=<\x11emD@',-72.819135,40.854647,sports_and_recreation,sports_and_recreation,[sports_and_recreation],0.949228,NaN,2025-06-25.0,sports_and_recreation
3952,1dc16ffd-4216-49f3-8d30-cfe6f1bf466a,b'\x01\x01\x00\x00\x00\xa7AM\x04\x1cxR\xc0R\x9e7\x1d\x1aJE@',-73.876710,42.578922,environmental_or_ecological_service,environmental_conservation_organization,"[services_and_business, environmental_or_ecological_service, environmental_conservation_organization]",0.919912,NaN,2025-06-25.0,services_and_business
2615,fd8b20c3-2168-429a-9234-f728f4e02332,b'\x01\x01\x00\x00\x00\xac-D\xa0];R\xc0\xfbF\xcf\x88-yD@',-72.927589,40.946702,sport_or_fitness_facility,scuba_diving_center,"[sports_and_recreation, sport_or_fitness_facility, diving_center, scuba_diving_center]",0.919912,NaN,2025-06-25.0,sports_and_recreation
1030,fcb17403-b755-4525-a90a-bdd97baa521b,b'\x01\x01\x00\x00\x00\x05\xcd\x9dOS*R\xc0\x16\xd04\xf6_uD@',-72.661335,40.916991,restaurant,spanish_restaurant,"[food_and_drink, restaurant, european_restaurant, iberian_restaurant, spanish_restaurant]",0.990219,NaN,2025-06-25.0,food_and_drink
3165,98871729-6b98-4d3d-9df9-c40ec33e271a,"b""\x01\x01\x00\x00\x00\x86V'g((R\xc0\xd5\xcd\xc5\xdf\xf6xD@""",-72.627466,40.945034,wellness_service,massage_therapy,"[lifestyle_services, wellness_service, massage_therapy]",0.919912,NaN,2025-06-25.0,lifestyle_services
388,7b546a4f-2e64-4e23-9049-d33ca4ab36f1,b'\x01\x01\x00\x00\x00b1\x870\x80-R\xc0\xde\xc4\xd2\xb8\x9euD@',-72.710949,40.918906,fashion_and_apparel_store,shoe_store,"[shopping, fashion_and_apparel_store, shoe_store]",0.990219,UGG,2025-06-25.0,shopping
263,c23734eb-934c-4316-b7e3-36bc23e9e4cf,b'\x01\x01\x00\x00\x00\x13\xd7\x8cg\xeb5R\xc0\xa7\\\x02l\x17mD@',-72.842493,40.852277,home_service,contractor,"[services_and_business, home_service, contractor]",0.919912,NaN,2025-06-25.0,services_and_business


In [14]:
overture_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_overture.parquet"

overture_pq = pq.ParquetFile(overture_file_path)
overture_first_100_rows = next(overture_pq.iter_batches(batch_size=100))
overture_df = pa.Table.from_batches([overture_first_100_rows]).to_pandas()

In [15]:
overture_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,commercial_places_within_1_mi,community_places_within_1_mi
23,106447,STUDENT FINEST CONVENIENT,22 MAIN ST,CORTLAND,13045,0,CONV. STORE,-76.18044,42.60048,b'\x01\x01\x00\x00\x00\xfa\xb86T\x8c\x0bS\xc0w\xf3T\x87\xdcLE@',360239706002,36023970600,160,64
47,69008,LITTLE ITALY VARIETY STORE,288 GLEN STREET,GLEN COVE,11542,0,STATIONERY-CARD,-73.62098,40.85905,"b'\x01\x01\x00\x00\x00\x16\xde\xe5""\xbegR\xc0}\xd0\xb3Y\xf5mD@'",360595172022,36059517202,212,58
90,96572,SX GOURMET CORP,115 RIVINGTON ST,NEW YORK,10002,0,DELI-GROCERY,-73.98771,40.71988,b'\x01\x01\x00\x00\x00\xa9\xfb\x00\xa46\x7fR\xc0\xb2\x85 \x07%\\D@',360610018003,36061001800,4980,728
41,65525,SANDY'S CARD GIFT & STATIONERY,1355 E MONTAUK HWY #9,MASTIC,11950,0,STATIONERY-CARD,-72.85039,40.80482,b'\x01\x01\x00\x00\x00\x1c\xb6-\xcal6R\xc0W\x95}W\x04gD@',361031594164,36103159416,145,11
51,111902,727 BRENTWOOD LIQUORS INC,727 SUFFOLK AVE,BRENTWOOD,11717,0,LIQUOR,-73.24848,40.78143,b'\x01\x01\x00\x00\x00rm\xa8\x18\xe7OR\xc0\x80\x0e\xf3\xe5\x05dD@',361031456041,36103145604,130,38


## LEHD Data

In [16]:
rac_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_rac.parquet"

rac_pq = pq.ParquetFile(rac_file_path)
rac_first_100_rows = next(rac_pq.iter_batches(batch_size=100))
rac_df = pa.Table.from_batches([rac_first_100_rows]).to_pandas()

In [17]:
rac_df.sample(2)

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,region,competitors_within_1_mi,rac_c000,rac_ce01,rac_ce02,rac_ce03,rac_ce01_prop,rac_ce02_prop
19,108348,D&H GAS CO INC,1009 ROUTE 109,FARMINGDALE,11735,independent,CONV. STORE,-73.43210,40.72465,b'\x01\x01\x00\x00\x00\x81&\xc2\x86\xa7[R\xc0\xf0\x85\xc9T\xc1\\D@',361031223001,36103122300,Long Island,26,1320.0,198.0,308.0,814.0,0.150000,0.233333
20,110259,ALMOST FREE,500 PEARL ST LOBBY,NEW YORK,10007,chain,CONV. STORE,-74.00135,40.71355,b'\x01\x01\x00\x00\x00`vO\x1e\x16\x80R\xc0\xc8\x07=\x9bU[D@',360610029011,36061002901,New York City,192,284.0,49.0,60.0,175.0,0.172535,0.211268


In [18]:
wac_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_wac.parquet"

wac_pq = pq.ParquetFile(wac_file_path)
wac_first_100_rows = next(wac_pq.iter_batches(batch_size=100))
wac_df = pa.Table.from_batches([wac_first_100_rows]).to_pandas()

In [19]:
wac_df.sample(2)

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,competitors_within_1_mi,wac_cns01,wac_cns02,wac_cns03,wac_cns04,wac_cns05,wac_cns06,wac_cns07,wac_cns08,wac_cns09,wac_cns10,wac_cns11,wac_cns12,wac_cns13,wac_cns14,wac_cns15,wac_cns16,wac_cns17,wac_cns18,wac_cns19,wac_cns20,wac_c000,wac_ce01,wac_ce02,wac_ce03,wac_ce01_prop,wac_ce02_prop
78,79240,LA ROSA WINES & LIQUORS,3906 BROADWAY,NEW YORK,10032,independent,LIQUOR,-73.94190,40.83796,b'\x01\x01\x00\x00\x00\x8e\x06\xf0\x16H|R\xc0\x97\xad\xf5EBkD@',360610245001,36061024500,131,0.0,0.0,0.0,0.0,0.0,7.0,29.0,7.0,0.0,0.0,21.0,3.0,130.0,4.0,0.0,375.0,0.0,46.0,0.0,0.0,622.0,64.0,115.0,443.0,0.102894,0.184887
4,108863,FQ MULTI-SERVICE,1702 PALMETTO STREET,RIDGEWOOD,11385,independent,GROCERY STORE,-73.90473,40.70571,b'\x01\x01\x00\x00\x00rm\xa8\x18\xe7yR\xc0\xe7:\x8d\xb4TZD@',360810589003,36081058900,159,0.0,0.0,0.0,3.0,7.0,0.0,2.0,0.0,2.0,0.0,0.0,0.0,0.0,18.0,0.0,0.0,0.0,0.0,3.0,0.0,35.0,11.0,10.0,14.0,0.314286,0.285714


## AADT Data

In [20]:
#aadt_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/data_aadt_2025.parquet"

#aadt_pq = pq.ParquetFile(aadt_file_path)
#aadt_first_100_rows = next(aadt_pq.iter_batches(batch_size=100))
#aadt_df = pa.Table.from_batches([aadt_first_100_rows]).to_pandas()

aadt_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/data_aadt_2025.parquet"
aadt_df = pd.read_parquet(str(aadt_file_path))

In [21]:
print(aadt_df['aadt_count'].isna().sum(), len(aadt_df))
aadt_df.sample(10)

33 16793


,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,bg_geoid,tract_geoid,competitors_within_1_mi,index_right,aadt_count,aadt_road_name,aadt_functional_class,aadt_lanes,Rmp,dist_to_read_m
7419,115637,965 MORRIS PARK DELI GROCERY C,965 MORRIS PARK AVE,BRONX,10462,independent,DELI-GROCERY,-73.85730,40.84763,360050244002,36005024400,110,230.0,10665.0,MORRIS PARK AVE,16,6.0,,1.453927
4821,64279,WESTERN BEEF #14,1717 DUTCH BROADWAY,ELMONT,11003,chain,SUPERMARKETS,-73.70549,40.68809,360594098003,36059409800,28,5735.0,14248.0,DUTCH BROADWAY,16,4.0,,17.356782
12799,77729,ONE STOP MINI MART,51 LIBERTY ST,BATH,14810,independent,CONV. STORE,-77.31823,42.33518,361019611003,36101961100,8,37874.0,5250.0,,16,2.0,,0.659858
1367,99030,CHUNNU NEWS INC,700 3RD AVE,NEW YORK,10017,independent,NEWS-SMOKE,-73.97359,40.75195,360610092001,36061009200,202,6236.0,25442.0,3RD AVE,14,15.0,,0.418519
11902,112704,PLAYERS CAFE,241 BLEECKER ST,NEW YORK,10014,independent,CONV. STORE,-74.00269,40.73075,360610067001,36061006700,114,6576.0,6114.0,BLEECKER ST,16,1.0,,2.397174
10988,77807,MIRABITO #84-BINGHAMTON,215 CLINTON ST,BINGHAMTON,13905,chain,CONV. STORE,-75.92986,42.10669,360070002001,36007000200,19,51087.0,7059.0,CLINTON ST,17,2.0,,2.910976
3577,104730,EAST SIDE SMOKE SHOP,1390 3RD AVE,NEW YORK,10075,independent,NEWS-SMOKE,-73.95718,40.77443,360610138009,36061013800,83,7088.0,16862.0,3RD AVE,14,5.0,,1.579424
7253,74651,SPEEDY'S 102,2634 ST RT 49,WEST MONROE,13167,independent,CONV. STORE,-76.09425,43.28574,360750207041,36075020704,0,24559.0,10877.0,,06,6.0,,20.535820
16151,107045,ADK RIVERSIDE LLC,1580 SOUTH MAIN ST,SAINT REGIS FALLS,12980,independent,RESTAURANT-HOTEL,-74.54504,44.67278,360339507011,36033950701,1,39833.0,1233.0,,07,2.0,,0.258947
16658,112969,KHUSDIL INC,310 RIGOR HILL RD,CHATHAM,12037,independent,CONV. STORE,-73.58143,42.29129,360210008002,36021000800,0,42291.0,268.0,RIGOR HILL RD,09,2.0,,0.413126


## EPA Data

In [22]:
epa_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_epa.parquet"

epa_pq = pq.ParquetFile(epa_file_path)
epa_first_100_rows = next(epa_pq.iter_batches(batch_size=100))
epa_df = pa.Table.from_batches([epa_first_100_rows]).to_pandas()

In [23]:
epa_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,competitors_within_1_mi,OBJECTID,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,D1A,D1B,D1C,D1C8_RET,D1C8_OFF,D1C8_IND,D1C8_SVC,D1C8_ENT,D1C8_ED,D1C8_HLTH,D1C8_PUB,D3A,NatWalkInd,STATEFP2,COUNTYFP2,TRACTCE2,BLKGRPCE2
52,96365,RITE AID #4277,115-10 MERRICK BLVD,JAMAICA,11434,chain,DRUG STORE,-73.77773,40.68961,b'\x01\x01\x00\x00\x00\x8f\xc7\x0cT\xc6qR\xc0N\x7f\xf6#EXD@',360810280001,36081028000,56,149545,36,81,28000,1,13.261611,43.039066,10.790351,0.685102,0.024468,0.000000,9.958442,0.000000,0.000000,0.122340,0.000000,30.202323,14.166667,36,081,028000,1
84,113405,MOONLIGHT DELI LLC,2829 JAMES ST,SYRACUSE,13206,independent,CONV. STORE,-76.10659,43.06886,b'\x01\x01\x00\x00\x00)\x05\xdd^\xd2\x06S\xc0T\x00\x8cg\xd0\x88E@',360670010002,36067001000,21,158371,36,67,1000,2,7.040032,16.413484,8.445387,0.291677,6.522968,0.026516,0.159097,0.490548,0.000000,0.954581,0.000000,28.352455,14.833333,36,067,001000,2
3,105306,MOBIL MART,380 COMMACK ROAD,DEER PARK,11729,independent,CONV. STORE,-73.31394,40.75946,b'\x01\x01\x00\x00\x00\x00:\xcc\x97\x17TR\xc0aO;\xfc5aD@',361031228012,36103122801,33,160168,36,103,122801,2,2.812744,7.365136,4.836029,0.704368,0.401821,2.798563,0.364002,0.543640,0.000000,0.023637,0.000000,18.498692,10.166667,36,103,122801,2
87,101053,ALITON'S PHARMACY HOME HEALTH,10-12 SUSSEX ST,PORT JERVIS,12771,independent,DRUG STORE,-74.69290,41.37437,b'\x01\x01\x00\x00\x00\x7f\xd9=yX\xacR\xc0@M-[\xeb\xafD@',360710022001,36071002200,8,150872,36,71,2200,1,3.630945,5.831016,5.111444,0.752656,0.380463,0.521070,1.116578,0.926346,0.000000,0.231586,1.182745,27.437119,18.333333,36,071,002200,1
10,79722,KANG MEI QIN INC,5101 6TH AVE,BROOKLYN,11220,independent,GROCERY STORE,-74.00878,40.64313,b'\x01\x01\x00\x00\x00\xf86\xfd\xd9\x8f\x80R\xc0\xca\x89v\x15RRD@',360470098003,36047009800,166,154671,36,47,9800,3,44.589706,187.791261,11.790547,2.250923,0.428747,0.643121,2.465296,2.036549,1.607802,2.358109,0.000000,23.053079,11.333333,36,047,009800,3


## ACS Data

In [24]:
acs_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_acs.parquet"

acs_pq = pq.ParquetFile(acs_file_path)
acs_first_100_rows = next(acs_pq.iter_batches(batch_size=100))
acs_df = pa.Table.from_batches([acs_first_100_rows]).to_pandas()

In [25]:
acs_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,median_household_income,households_total,total_population,age_18_29_proportion,age_30_64_proportion,age_65_plus_proportion,poverty_rate,bachelors_degree_proportion_25plus,post_grad_degree_proportion_25plus,labor_force_participation_rate,unemployment_rate,owner_occupied_proportion,renter_occupied_proportion,family_households_proportion,nonfamily_multiperson_households_proportion,living_alone_households_proportion,no_vehicle_households_proportion
63,100319,MIA TRAVEL MULTISERVICE,5 E CLARK PLACE,BRONX,10452,0,TRAVEL AGENT,-73.91877,40.83828,b'\x01\x01\x00\x00\x00\xf0\xa2\xaf \xcdzR\xc0\x08rP\xc2LkD@',360050221013,36005022101,65609.0,625.0,1319.0,0.388931,0.595906,0.015163,0.319920,0.143275,0.000000,0.653651,0.154867,0.000000,1.000000,0.872000,0.060800,0.067200,0.721600
13,88345,MASTER DELI INC,2009 EASTERN PKWY,BROOKLYN,11233,0,DELI-GROCERY,-73.90576,40.67988,"b'\x01\x01\x00\x00\x00\x99\x81\xca\xf8\xf7yR\xc0,\x9a\xceN\x06WD@'",360470369001,36047036900,NaN,596.0,939.0,0.171459,0.667732,0.160809,0.320350,0.064764,0.203074,0.576642,0.097649,0.192953,0.807047,0.416107,0.053691,0.530201,0.701342
61,94655,WILLY'S DELI CORP,1356 ROCKAWAY PKWY,BROOKLYN,11236,0,DELI-GROCERY,-73.90388,40.64652,b'\x01\x01\x00\x00\x00\x8b\xe0\x7f+\xd9yR\xc0\xdf\x1a\xd8*\xc1RD@',360470968001,36047096800,105292.0,195.0,486.0,0.102881,0.594650,0.302469,0.053398,0.180258,0.199571,0.475149,0.050209,0.687179,0.312821,0.841026,0.061538,0.097436,0.220513
11,79722,KANG MEI QIN INC,5101 6TH AVE,BROOKLYN,11220,0,GROCERY STORE,-74.00878,40.64313,b'\x01\x01\x00\x00\x00\xf86\xfd\xd9\x8f\x80R\xc0\xca\x89v\x15RRD@',360470098003,36047009800,56522.0,442.0,1260.0,0.238889,0.550000,0.211111,0.364087,0.142857,0.011583,0.593130,0.097812,0.099548,0.900452,0.834842,0.165158,0.000000,0.601810
95,112326,HARTSDALE TOBACCO & CANDY CORP,205 E HARTSDALE AVE,HARTSDALE,10530,0,CONV. STORE,-73.79610,41.01232,b'\x01\x01\x00\x00\x00\x1e\x16jM\xf3rR\xc0\x1a\x8b\xa6\xb3\x93\x81D@',361190108041,36119010804,108958.0,767.0,1447.0,0.070491,0.536973,0.392536,0.035932,0.352597,0.381858,0.618656,0.041020,0.885267,0.114733,0.629726,0.009126,0.361147,0.067797


## Master Data Sample

In [26]:
master_file_path = Path(".").resolve().parent / "data/final/master_data.parquet"
master_df = pd.read_parquet(str(master_file_path))

print(master_df.shape)
print(master_df.columns)
master_df.sample(5)

(14341, 58)
Index(['agtno', 'avg_daily_sales', 'chain', 'bustype', 'longitude', 'latitude',
       'bg_geoid', 'tract_geoid', 'region', 'competitors_within_1_mi',
       'aadt_count', 'aadt_functional_class', 'aadt_lanes', 'dist_to_read_m',
       'median_household_income', 'total_population', 'age_18_29_proportion',
       'age_30_64_proportion', 'poverty_rate',
       'bachelors_degree_proportion_25plus',
       'post_grad_degree_proportion_25plus', 'labor_force_participation_rate',
       'unemployment_rate', 'owner_occupied_proportion',
       'family_households_proportion', 'no_vehicle_households_proportion',
       'D1B', 'D1C', 'D3A', 'NatWalkInd', 'commercial_places_within_1_mi',
       'community_places_within_1_mi', 'rac_c000', 'rac_ce01_prop',
       'rac_ce02_prop', 'wac_cns01', 'wac_cns02', 'wac_cns03', 'wac_cns04',
       'wac_cns05', 'wac_cns06', 'wac_cns07', 'wac_cns08', 'wac_cns09',
       'wac_cns10', 'wac_cns11', 'wac_cns12', 'wac_cns13', 'wac_cns14',
       'wac_cns

,agtno,avg_daily_sales,chain,bustype,longitude,latitude,bg_geoid,tract_geoid,region,competitors_within_1_mi,aadt_count,aadt_functional_class,aadt_lanes,dist_to_read_m,median_household_income,total_population,age_18_29_proportion,age_30_64_proportion,poverty_rate,bachelors_degree_proportion_25plus,post_grad_degree_proportion_25plus,labor_force_participation_rate,unemployment_rate,owner_occupied_proportion,family_households_proportion,no_vehicle_households_proportion,D1B,D1C,D3A,NatWalkInd,commercial_places_within_1_mi,community_places_within_1_mi,rac_c000,rac_ce01_prop,rac_ce02_prop,wac_cns01,wac_cns02,wac_cns03,wac_cns04,wac_cns05,wac_cns06,wac_cns07,wac_cns08,wac_cns09,wac_cns10,wac_cns11,wac_cns12,wac_cns13,wac_cns14,wac_cns15,wac_cns16,wac_cns17,wac_cns18,wac_cns19,wac_cns20,wac_c000,wac_ce01_prop,wac_ce02_prop
2881,109228,1042.238208,chain,CONV. STORE,-77.64594,42.99666,360550133003,36055013300,Finger Lakes,2.0,9521.0,16,2.0,26.018002,110938.0,1183.0,0.039730,0.548605,0.027756,0.294894,0.189261,0.586379,0.015581,0.901503,0.585977,0.00000,0.174476,0.030023,2.796680,5.333333,7.0,4.0,670.0,0.182090,0.161194,11.0,0.0,0.0,4.0,0.0,50.0,16.0,10.0,9.0,0.0,2.0,31.0,0.0,22.0,79.0,63.0,0.0,15.0,28.0,64.0,404.0,0.257426,0.173267
7830,19942,262.559066,independent,DELI-GROCERY,-74.40924,41.27731,360710145013,36071014501,Mid-Hudson,0.0,3871.0,17,2.0,48.291606,120441.0,1091.0,0.064161,0.622365,0.000000,0.260994,0.110899,0.655752,0.000000,0.923554,0.795455,0.00000,0.312656,0.045290,3.245982,4.500000,6.0,0.0,651.0,0.195084,0.216590,52.0,0.0,0.0,41.0,75.0,22.0,22.0,0.0,30.0,3.0,1.0,11.0,0.0,8.0,0.0,0.0,0.0,22.0,3.0,0.0,290.0,0.189655,0.289655
6470,115165,209.367123,independent,LIQUOR,-72.91494,40.79911,361031591064,36103159106,Long Island,5.0,12103.0,16,6.0,5.424304,115063.0,1070.0,0.182243,0.601869,0.002985,0.195631,0.094340,0.746076,0.039604,1.000000,0.947743,0.00000,NaN,NaN,NaN,NaN,10.0,8.0,803.0,0.159402,0.242839,0.0,0.0,0.0,2.0,4.0,12.0,263.0,14.0,0.0,3.0,6.0,3.0,0.0,0.0,7.0,27.0,0.0,13.0,2.0,0.0,356.0,0.221910,0.441011
9705,49963,2471.872603,chain,SUPERMARKETS,-76.86764,42.36972,360979504004,36097950400,Southern Tier,5.0,9472.0,04,2.0,0.425230,44250.0,464.0,0.040948,0.534483,0.121442,0.140940,0.149888,0.520747,0.067729,0.627832,0.443366,0.07767,1.991257,3.560404,8.074073,9.666667,90.0,18.0,288.0,0.180556,0.319444,0.0,0.0,0.0,19.0,0.0,0.0,273.0,64.0,0.0,0.0,0.0,7.0,0.0,0.0,193.0,103.0,0.0,55.0,8.0,287.0,1009.0,0.163528,0.380575
13981,96986,4484.050685,independent,GAS,-73.93458,40.73967,360810199022,36081019902,New York City,64.0,33612.0,14,2.0,0.498539,NaN,8.0,1.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.00000,NaN,NaN,NaN,NaN,727.0,179.0,6.0,0.333333,0.000000,0.0,0.0,8.0,695.0,339.0,566.0,34.0,145.0,428.0,37.0,78.0,335.0,146.0,317.0,115.0,122.0,0.0,48.0,98.0,407.0,3918.0,0.079632,0.126340


## Missingness and Variance Check

In [27]:
def summarize_numeric_features(df, numeric_cols):

    summary = pd.DataFrame({
        "pct_missing": df[numeric_cols].isna().mean() * 100,
        "pct_zero": (df[numeric_cols] == 0).mean() * 100,
        "pct_mode": [df[col].value_counts(normalize=True, dropna=True).max() * 100 for col in numeric_cols],
        "mode": [df[col].value_counts(dropna=True).idxmax() if df[col].notna().any() else pd.NA for col in numeric_cols],
        "mean": df[numeric_cols].mean(),
        "std": df[numeric_cols].std(),
        "iqr": df[numeric_cols].quantile(0.75) - df[numeric_cols].quantile(0.25),
        "cv": (df[numeric_cols].std()) / (df[numeric_cols].mean().abs()),
        "n_unique": df[numeric_cols].nunique()
    })

    return summary.sort_values("pct_missing", ascending=False)

def summarize_categorical_features(df, categorical_cols):
    results = {}

    for col in categorical_cols:

        pct_missing = df[col].isna().mean() * 100

        distribution = df[col].value_counts(dropna=False, normalize=True).mul(100).rename("pct").to_frame()

        results[col] = {"pct_missing": pct_missing, "distribution": distribution}

    return results

In [28]:
numeric_features = ['competitors_within_1_mi', 'aadt_count', 'aadt_lanes', 'dist_to_read_m',
       'median_household_income', 'total_population', 'age_18_29_proportion', 'age_30_64_proportion',
       'poverty_rate', 'bachelors_degree_proportion_25plus', 'post_grad_degree_proportion_25plus',
       'labor_force_participation_rate', 'unemployment_rate', 'owner_occupied_proportion',
       'family_households_proportion', 'no_vehicle_households_proportion',
       'D1B', 'D1C', 'D3A', 'NatWalkInd', 'commercial_places_within_1_mi',
       'community_places_within_1_mi', 'rac_c000', 'rac_ce01_prop',
       'rac_ce02_prop', 'wac_cns01', 'wac_cns02', 'wac_cns03', 'wac_cns04',
       'wac_cns05', 'wac_cns06', 'wac_cns07', 'wac_cns08', 'wac_cns09',
       'wac_cns10', 'wac_cns11', 'wac_cns12', 'wac_cns13', 'wac_cns14',
       'wac_cns15', 'wac_cns16', 'wac_cns17', 'wac_cns18', 'wac_cns19',
       'wac_cns20', 'wac_c000', 'wac_ce01_prop', 'wac_ce02_prop']

categorical_features = ['chain', 'bustype', 'aadt_functional_class']

numeric_summary = summarize_numeric_features(master_df, numeric_features)
categorical_summary = summarize_categorical_features(master_df, categorical_features)

numeric_summary

,pct_missing,pct_zero,pct_mode,mode,mean,std,iqr,cv,n_unique
NatWalkInd,18.290217,0.000000,2.227343,16.000000,12.550549,4.124931,6.666667,0.328665,114
D3A,18.290217,0.000000,0.128008,27.524029,21.927281,12.297690,17.052359,0.560840,6207
D1C,18.290217,0.146433,0.179211,0.000000,28.550338,136.314702,10.901196,4.774539,6192
D1B,18.290217,0.216163,0.264550,0.000000,40.631699,63.115681,53.089343,1.553361,6186
median_household_income,10.424657,0.000000,1.370076,250001.000000,92799.536899,47438.643700,58289.750000,0.511195,5833
family_households_proportion,1.317900,0.160379,0.847937,1.000000,0.612610,0.194447,0.265905,0.317408,6761
no_vehicle_households_proportion,1.317900,11.937801,12.097230,0.000000,0.261447,0.280911,0.399796,1.074447,5922
poverty_rate,1.317900,5.892197,5.970888,0.000000,0.143138,0.140365,0.156161,0.980628,6685
owner_occupied_proportion,1.317900,5.348302,5.419729,0.000000,0.536270,0.317095,0.570462,0.591298,6339
unemployment_rate,1.241197,17.662646,17.884629,0.000000,0.065368,0.074246,0.075611,1.135812,5357


In [29]:
categorical_summary['chain']

{'pct_missing': np.float64(0.0),
 'distribution':                    pct
 chain                 
 independent  73.579248
 chain        26.420752}

## Correlation

In [30]:
correlation_file_path = Path(".").resolve().parent / "src/correlation_test/high_correlation_test.csv"
correlation_df = pd.read_csv(str(correlation_file_path))

correlation_df

,variable_1,variable_2,correlation,abs_correlation
0,owner_occupied_proportion,renter_occupied_proportion,-1.000,1.000
1,wac_c000,wac_ce03,0.974,0.974
2,arts_and_entertainment_within_1_mi,shopping_within_1_mi,0.971,0.971
3,shopping_within_1_mi,restaurant_within_1_mi,0.970,0.970
4,shopping_within_1_mi,social_or_community_service_within_1_mi,0.968,0.968
5,restaurant_within_0.5_mi,restaurant_within_1_mi,0.965,0.965
6,arts_and_entertainment_within_1_mi,social_or_community_service_within_1_mi,0.962,0.962
7,arts_and_entertainment_within_1_mi,restaurant_within_1_mi,0.955,0.955
8,restaurant_within_1_mi,bar_within_1_mi,0.953,0.953
9,arts_and_entertainment_within_0.5_mi,arts_and_entertainment_within_1_mi,0.951,0.951


## PCA Data

In [31]:
pca_train_file_path = Path(".").resolve().parent / "data/modeling/pca/X_pca_train.parquet"
pca_df = pd.read_parquet(str(pca_train_file_path))

print(pca_df.shape)
print(pca_df.columns)
pca_df.sample(5)

(8669, 18)
Index(['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10',
       'PC11', 'PC12', 'PC13', 'PC14', 'PC15', 'PC16', 'PC17', 'agtno'],
      dtype='str')


,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11,PC12,PC13,PC14,PC15,PC16,PC17,agtno
2814,0.028394,0.395168,1.911178,-0.551681,-0.472338,0.680768,-1.222844,0.794011,-0.388497,-1.961498,-1.660499,-0.025217,0.604225,0.535091,0.049790,-0.051998,0.284943,112575
1230,-2.952333,-0.118574,-0.793954,-1.846201,1.177825,0.812116,1.188990,-0.177347,-0.158441,-0.518332,1.086651,0.559186,0.557783,0.167826,0.407860,0.299042,-0.471628,107149
124,1.075009,-1.353118,-1.341559,1.167213,-1.242210,-2.013974,0.250436,-0.753897,-0.602246,0.420516,0.478473,0.048874,-0.392518,-0.021192,0.015550,-0.679831,-0.027195,100933
1893,0.544421,0.604836,0.977061,-2.482978,0.403044,0.853594,0.277180,-0.624402,-0.359190,0.059725,-0.106887,0.048689,-0.034018,0.030917,0.152526,0.517181,0.254937,109785
4163,3.707824,-0.904453,-0.187596,1.959834,1.634690,-0.965248,0.672404,0.038969,-0.173099,0.672912,-0.214933,-0.372717,0.566935,-0.120577,-0.104593,0.175620,0.074562,11561


## GMM Data

In [32]:
gmm_train_file_path = Path(".").resolve().parent / "data/modeling/gmm/X_gmm_train.parquet"
gmm_df = pd.read_parquet(str(gmm_train_file_path))

print(gmm_df.shape)
print(gmm_df.columns)
gmm_df.sample(5)

(8669, 8)
Index(['agtno', 'gmm_prob_0', 'gmm_prob_1', 'gmm_prob_2', 'gmm_prob_3',
       'gmm_prob_4', 'gmm_cluster', 'gmm_max_prob'],
      dtype='str')


,agtno,gmm_prob_0,gmm_prob_1,gmm_prob_2,gmm_prob_3,gmm_prob_4,gmm_cluster,gmm_max_prob
250,101655,0.933477,0.000165,0.026696,0.000400,0.039263,0,0.933477
5166,36978,0.612369,0.006123,0.050360,0.009674,0.321474,0,0.612369
8546,98145,0.475646,0.000285,0.069973,0.000028,0.454068,0,0.475646
2554,111879,0.097953,0.023440,0.000151,0.876582,0.001874,3,0.876582
2510,111706,0.010440,0.163098,0.000785,0.777876,0.047800,3,0.777876
